# Imports

In [ ]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import LineString
import matplotlib.patheffects as pe
import fiona
from shapely.geometry import LineString, MultiLineString
import math

# Standardize figure formats

In [ ]:
mm = 1/25.4
mpl.rcParams.update({
    "figure.dpi": 300,          # on-screen
    "savefig.dpi": 600,         # export
    "font.family": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 6.5,           # ~6–7 pt at final size
    "axes.titlesize": 7,
    "axes.labelsize": 6.5,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
})
def add_scale_bar(ax, length_km=20, location=(0.9, 0.8), linewidth=1, tick_height=0.01, label_offset=0.02, km_offset=0.01):
    x, y = location  # Adjusted location (higher y value to move the scale bar upward)
    bar_half_length = 0.05  # Half the scale bar length in axes fraction

    # Draw the scale bar
    ax.plot(
        [x - bar_half_length, x + bar_half_length], [y, y],  # Scale bar endpoints
        transform=ax.transAxes, color='black', linewidth=linewidth
    )

    # Draw perpendicular tick marks
    tick_positions = [x - bar_half_length, x, x + bar_half_length]
    for pos in tick_positions:
        ax.plot(
            [pos, pos], [y - tick_height / 2, y + tick_height / 2],  # Vertical line for ticks
            transform=ax.transAxes, color='black', linewidth=linewidth
        )

    # Add numeric labels below the tick marks
    ax.text(
        x - bar_half_length, y - tick_height - label_offset, "0", transform=ax.transAxes, 
        ha='center', va='center', fontsize=6
    )
    ax.text(
        x, y - tick_height - label_offset, f"{length_km // 2}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=6
    )
    ax.text(
        x + bar_half_length, y - tick_height - label_offset, f"{length_km}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=6
    )

    # Add "km" label slightly to the right of the scale bar
    ax.text(
        x + bar_half_length + km_offset, y, "km", transform=ax.transAxes, 
        ha='left', va='center', fontsize=7
    )

def add_north_arrow(ax, location=(0.9, 0.87), size=0.05, fontsize=7, label_offset=0.03):
    """
    Add a north arrow to the plot, with "N" positioned slightly above the arrow.
    """
    x, y = location

    # Draw the arrow
    ax.annotate(
        '', xy=(x, y + size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', headwidth=6, headlength=6, width=2.5)
    )

    # Add the "N" label slightly above the arrow
    ax.text(
        x, y + size + label_offset, "N", transform=ax.transAxes,
        fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black"
    )


In [ ]:


def longest_linestring(geom):
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, LineString):
        return geom
    if isinstance(geom, MultiLineString):
        return max(geom.geoms, key=lambda g: g.length)
    if geom.geom_type == "GeometryCollection":
        lines = [g for g in geom.geoms if g.geom_type in ("LineString", "MultiLineString")]
        return max(lines, key=lambda g: g.length) if lines else None
    return None

def angle_at_fraction(ls, frac=0.5):
    if ls is None or ls.is_empty:
        return 0.0
    pt = ls.interpolate(frac, normalized=True)
    d  = ls.interpolate(min(frac + 1e-3, 1), normalized=True)
    return math.degrees(math.atan2(d.y - pt.y, d.x - pt.x))


# Paths and CRS

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
rivers_path = gpd.read_file(base_path / "dphil_common_cross_cutting/common_incoming_data/osm/gis_osm_waterways_free_1/gis_osm_waterways_free_1.shp")

simple_rivers_path = base_path / "dphil_common_cross_cutting/common_incoming_data/rivers/rivers.gpkg"
print("GPKG in drivers?", "GPKG" in fiona.supported_drivers)
print(fiona.supported_drivers.get("GPKG"))  # should show 'rw' or 'r'

jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"

out_dir = base_path / "dphil_paper_2/results/figures/baseline_figure"

output_dir = base_path / "dphil_paper_2/results"

jamaica_metric_grid_crs = "EPSG:3448"

catchments = gpd.read_file(base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg")
# catchments = gpd.read_file(catchments_unionized_final)
catchments.head()
print("Catchments:", len(catchments))


In [ ]:
catchment_uid_col = "catchment_uid"  # <-- update if different


### Read in boundary 

In [ ]:
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

## Read in admin boundary 

### Read in land use

In [ ]:
land_use = gpd.read_file(base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_LandCover.shp")
print(land_use.crs)

In [ ]:
land_use["area_m2"] = land_use.geometry.area
land_use["area_ha"] = land_use["area_m2"] / 1e4
print(f"Total area: {land_use['area_ha'].sum():,.2f} ha")

# River info

In [ ]:
rivers_simple = gpd.read_file(simple_rivers_path)  # add layer=... if needed


In [ ]:
# --- labels dataframe ---
labels = (
    rivers_simple.loc[rivers_simple["name"].notna() & (rivers_simple["name"].str.strip() != "")]
    .assign(len_m=lambda df: df.geometry.length)
    .sort_values("len_m", ascending=False)
    .drop_duplicates(subset="name")
    .copy()
)
labels["geom_ls"]     = labels.geometry.apply(longest_linestring)
labels["label_point"] = labels["geom_ls"].apply(lambda g: g.interpolate(0.5, normalized=True))
labels["angle"]       = labels["geom_ls"].apply(angle_at_fraction)
labels = labels[labels["geom_ls"].apply(lambda g: g.length >= 1500)].copy()

# --- plot ---
fig, ax = plt.subplots(figsize=(10, 10))

jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.8, zorder=1)

rivers_simple.plot(ax=ax, linewidth=0.6, color="#1f77b4", zorder=2)

for _, r in labels.iterrows():
    p = r["label_point"]
    ax.text(p.x, p.y, r["name"], fontsize=8,
            rotation=r["angle"], rotation_mode="anchor",
            ha="center", va="center", zorder=3,
            path_effects=[pe.withStroke(linewidth=2, foreground="white")])

# tighten to boundary so scalebar sizes sensibly
minx, miny, maxx, maxy = jamaica_boundary.total_bounds
ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)

add_scale_bar(ax)
add_north_arrow(ax)
title = "Rivers"  # pick your title

ax.set_axis_off()
plt.title(title, fontsize=20, fontweight='bold',
          fontname='Times New Roman', pad=20)
plt.tight_layout()

out_png = Path(out_dir) /"rivers.png"
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print(f"Saved {out_png}")
plt.show()
ax.set_title("Simple Rivers with Labels, Jamaica")
ax.set_axis_off()
plt.tight_layout()
plt.show()

# Rivers simple by catchments

In [ ]:
rivers_simple.head()

river_name_col = "name"              # <-- update if different


In [ ]:
# Clip rivers to catchments (line segments inside each polygon)
intersections = gpd.overlay(
    rivers_simple[[river_name_col, "geometry"]],
    catchments[[catchment_uid_col, "geometry"]],
    how="intersection",
    keep_geom_type=True
)

In [ ]:
# Length of each river segment inside each catchment
intersections["seg_len_m"] = intersections.geometry.length

# For each catchment, pick the river with the longest intersecting length
best_river_by_catchment = (
    intersections.sort_values("seg_len_m", ascending=False)
    .drop_duplicates(subset=[catchment_uid_col])
    [[catchment_uid_col, river_name_col, "seg_len_m"]]
)

# Optional: all rivers per catchment (ranked)
all_rivers_by_catchment = (
    intersections.groupby([catchment_uid_col, river_name_col], as_index=False)
    .agg(total_len_m=("seg_len_m", "sum"))
    .sort_values([catchment_uid_col, "total_len_m"], ascending=[True, False])
)

best_river_by_catchment





In [ ]:
best_river_by_catchment.to_csv(output_dir / "catchment_to_river_longest.csv", index=False)
all_rivers_by_catchment.to_csv(output_dir / "catchment_to_river_all.csv", index=False)

Rivers OSM by catchments

In [ ]:
catchment_uid_col = "catchment_uid"  # <-- update if different


rivers_path_osm = base_path / "dphil_common_cross_cutting/common_incoming_data/osm/gis_osm_waterways_free_1/gis_osm_waterways_free_1.shp"
rivers_osm = gpd.read_file(rivers_path_osm)

# --- enforce metric CRS for length ops (rivers_osm + catchments) ---
target_crs = "EPSG:3448"

if catchments.crs != target_crs:
    catchments = catchments.to_crs(target_crs)

if rivers_osm.crs != target_crs:
    rivers_osm = rivers_osm.to_crs(target_crs)

# ensure catchment_uid is a real column and clean
if catchment_uid_col not in catchments.columns:
    print("Columns:", catchments.columns.tolist())

catchments = catchments.reset_index()  # safe even if not needed
catchments.columns = [c.strip() for c in catchments.columns]


# Intersections (river segments inside catchments)
intersections = gpd.overlay(
    rivers_osm[[river_name_col, "geometry"]],
    catchments[[catchment_uid_col, "geometry"]],
    how="intersection",
    keep_geom_type=True
)
intersections["seg_len_m"] = intersections.geometry.length

# Total length per river (entire geometry)
river_total = (
    rivers_osm.assign(total_len_m=rivers_osm.geometry.length)
    [[river_name_col, "total_len_m"]]
)

# Sum segment lengths per catchment + river
catch_river = (
    intersections.groupby([catchment_uid_col, river_name_col], as_index=False)
    .agg(seg_len_m=("seg_len_m", "sum"))
    .merge(river_total, on=river_name_col, how="left")
)

# % of each river that lies inside each catchment
catch_river["river_pct_in_catchment"] = 100 * catch_river["seg_len_m"] / catch_river["total_len_m"]

# Total river length inside each catchment
catch_totals = (
    catch_river.groupby(catchment_uid_col, as_index=False)
    .agg(catchment_river_len_m=("seg_len_m", "sum"))
)

# % of each catchment’s river length contributed by each river
catch_river = catch_river.merge(catch_totals, on=catchment_uid_col, how="left")
catch_river["catchment_pct_by_river"] = 100 * catch_river["seg_len_m"] / catch_river["catchment_river_len_m"]



In [ ]:
catch_river

In [ ]:
# Export full catchment–river table
catch_river.to_csv(output_dir / "catchment_river_shares_osm.csv", index=False)
print("Saved:", output_dir / "catchment_river_shares_osm.csv")


In [ ]:
wm_units_path = base_path / "dphil_common_cross_cutting/common_incoming_data/rivers/water_management_units.gpkg"
wm_units = gpd.read_file(wm_units_path)  # add layer="..." if needed
wm_units.head()

In [ ]:
# Check columns to find the unit name field
print("WM unit columns:", wm_units.columns.tolist())
# Example: unit name column might be "name" or "wm_name"
wm_name_col = "Wmu_Name"  # <-- update after checking

# Ensure same CRS
target_crs = "EPSG:3448"
catchments = catchments.to_crs(target_crs)
wm_units = wm_units.to_crs(target_crs)

# Intersect catchments with WM units
intersections = gpd.overlay(
    catchments[[ "catchment_uid", "geometry"]],
    wm_units[[wm_name_col, "geometry"]],
    how="intersection",
    keep_geom_type=True
)

# Area of overlap
intersections["area_m2"] = intersections.geometry.area

# Pick the WM unit with the largest overlap per catchment
best_wm_by_catchment = (
    intersections.sort_values("area_m2", ascending=False)
    .drop_duplicates(subset=["catchment_uid"])
    [["catchment_uid", wm_name_col, "area_m2"]]
)

best_wm_by_catchment.head()


In [ ]:
# Total overlap area per catchment
catch_totals = (
    intersections.groupby("catchment_uid", as_index=False)
    .agg(total_overlap_m2=("area_m2", "sum"))
)

# All overlaps + share
all_wmu_by_catchment = intersections.merge(catch_totals, on="catchment_uid", how="left")
all_wmu_by_catchment["overlap_pct"] = 100 * all_wmu_by_catchment["area_m2"] / all_wmu_by_catchment["total_overlap_m2"]

# Export
all_wmu_by_catchment[["catchment_uid", wm_name_col, "area_m2", "overlap_pct"]].to_csv(
    output_dir / "catchment_to_wmu_all_overlaps.csv", index=False
)
print("Saved:", output_dir / "catchment_to_wmu_all_overlaps.csv")


In [ ]:
best_wm_by_catchment.to_csv(output_dir / "catchment_to_wmu.csv", index=False)
print("Saved:", output_dir / "catchment_to_wmu.csv")
